# GPOMDP vs NPG comparison

Loads `training_rewards.npz` from two or more separate `run.py` runs and overlays them with a 95% CI band.

This replaces `run.py`'s old `--algorithms gpomdp npg` comparison mode: each algorithm is now trained with its own
`run.py` invocation (see `--algorithm`), and the comparison plot is produced here instead.

In [4]:
import os
import sys

sys.path.append("..")

import numpy as np

from utils import plot_comparison

RecursionError: maximum recursion depth exceeded

Edit these to point at the output directories produced by your `run.py` runs, e.g.:

```bash
python3 run.py --env_id CartPole-v1 --algorithm gpomdp --output_dir runs/CartPole-v1/gpomdp
python3 run.py --env_id CartPole-v1 --algorithm npg    --output_dir runs/CartPole-v1/npg
```

In [ ]:
run_dirs = {
    "gpomdp": "../runs/CartPole-v1/gpomdp",
    "npg": "../runs/CartPole-v1/npg",
}
env_id = "CartPole-v1"
save_dir = "../runs/CartPole-v1"

In [ ]:
# rewards arrays are shape [1, n_iterations] for single-seed runs, or
# [n_seeds, n_iterations] for multiseed runs — plot_comparison handles both.
rewards_dict = {}

for algo, run_dir in run_dirs.items():
    npz_path = os.path.join(run_dir, "training_rewards.npz")
    if os.path.exists(npz_path):
        data = np.load(npz_path)
        rewards_dict[algo] = data["rewards"]
        print(f"{algo}: rewards {data['rewards'].shape}, seeds = {data['seeds']}")
    else:
        print(f"Warning: missing {npz_path} — skipping {algo}")

rewards_dict

In [ ]:
plot_comparison(rewards_dict, save_dir=save_dir, env_id=env_id)

## Single-run confidence interval

Load one run directory's `training_rewards.npz` (holding `rewards` of shape `[n_seeds, n_iterations]` and the matching `seeds` array, produced by a `--run_mode multiseed` run) and plot the mean with a 95% CI band across seeds. A single-seed run plots the mean only, with no band.

In [ ]:
single_run_dir = "../runs/CartPole-v1/gpomdp"

data = np.load(os.path.join(single_run_dir, "training_rewards.npz"))
curves = data["rewards"]
seeds = data["seeds"]
print(f"loaded curves {curves.shape} (n_seeds, n_iterations); seeds = {seeds}")

# Per-seed final performance, so you can spot seeds that under/over-performed.
for seed, curve in zip(seeds, curves):
    print(f"  seed {seed}: final = {curve[-1]:.2f}, max = {curve.max():.2f}")

plot_seed_ci(
    curves,
    save_path=os.path.join(single_run_dir, "training_rewards_ci.png"),
    title=f"Training reward across seeds — {env_id}",
)